# Benchmark CNN v3 / LWCNN and CNN-Transformer

This notebook produces the benchmarking numbers for the two models in your report:

1. **CNN v3 / LWCNN**
2. **CNN-Transformer**

It reports:

- trainable parameters
- total parameters
- checkpoint / saved-model size
- mean, median, p95 and p99 inference latency per **0.5-second audio window**
- windows per second
- real-time factor
- times faster than real time
- report-ready wording
- CNN-Transformer F1 score on internal and external test sets, when evaluation data paths are configured

The notebook can run in two ways:

- **Architecture benchmark:** builds the model architectures directly and benchmarks them. This is enough for latency and parameter count because trained weights do not change model speed.
- **Weights benchmark:** optionally loads your saved `.weights.h5` checkpoints if the architecture matches the saved checkpoint.
The F1 section is optional: if no internal/external test data paths are set, the benchmark still runs and the F1 columns are left as `NaN`.


## 1. Colab setup

Before running the notebook, enable GPU:

`Runtime` → `Change runtime type` → `Hardware accelerator` → `GPU`

Then run the cells below.

In [1]:
import os
import time
import json
import math
import glob
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Python:", platform.python_version())
print("GPU devices:", tf.config.list_physical_devices("GPU"))

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("Could not set memory growth:", e)

TensorFlow: 2.20.0
Python: 3.12.13
GPU devices: []


## 2. Mount Google Drive

This lets the notebook access saved checkpoints such as:

- `/content/drive/MyDrive/lwcnn_ext_search/...`
- `/content/drive/MyDrive/cnntransformer_ext_search/...`

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Benchmark configuration

The report describes the input as a log-mel spectrogram.  
Use the same input shape as your final models.

Your draft contains both `(64, 62, 1)` and `(64, 63, 1)` in different sections.  
The default below is `(64, 63, 1)`, but change it if your final model used `(64, 62, 1)`.

In [4]:
# === EDIT ONLY IF NEEDED ===
INPUT_SHAPE = (64, 63, 1)   # try (64, 62, 1) if this does not match your final model
WINDOW_DURATION_SECONDS = 0.5

# Streaming deployment should be benchmarked with batch size 1.
BATCH_SIZE = 1
WARMUP_RUNS = 30
MEASURE_RUNS = 300

# Optional checkpoint paths.
# These are likely candidates based on your Drive folders.
# If loading fails, leave them as None and use the architecture benchmark.
CNN_V3_WEIGHTS_PATH = "/content/drive/MyDrive/lwcnn_ext_search/trial_4/checkpoint.weights.h5"
CNN_TRANSFORMER_WEIGHTS_PATH = "/content/drive/MyDrive/cnntransformer_ext_search/trial_4/checkpoint.weights.h5"

# Optional full-model paths, if you have saved .keras or full .h5 models.
# Leave as None unless you know the model was saved with model.save(...).
CNN_V3_FULL_MODEL_PATH = None
CNN_TRANSFORMER_FULL_MODEL_PATH = None

# ---------------------------------------------------------------------
# Optional CNN-Transformer F1 evaluation paths
# ---------------------------------------------------------------------
# To compute internal/external F1, set either the NPZ path or the X/Y .npy paths
# for each split. The label convention assumed here is: 1 = Drone, 0 = No-drone.
# Supported NPZ key pairs include X/y, X_test/y_test, features/labels, or arr_0/arr_1.
CNN_TRANSFORMER_INTERNAL_TEST_NPZ = None
CNN_TRANSFORMER_EXTERNAL_TEST_NPZ = None

CNN_TRANSFORMER_INTERNAL_X_PATH = None
CNN_TRANSFORMER_INTERNAL_Y_PATH = None
CNN_TRANSFORMER_EXTERNAL_X_PATH = None
CNN_TRANSFORMER_EXTERNAL_Y_PATH = None

# Optional image-folder fallback, e.g. folders containing class subfolders.
# Use this only if your test data is stored as spectrogram images.
CNN_TRANSFORMER_INTERNAL_DATA_DIR = None
CNN_TRANSFORMER_EXTERNAL_DATA_DIR = None

CNN_TRANSFORMER_EVAL_BATCH_SIZE = 32
CNN_TRANSFORMER_DECISION_THRESHOLD = 0.50
CNN_TRANSFORMER_FLIP_INTERNAL_LABELS = False
CNN_TRANSFORMER_FLIP_EXTERNAL_LABELS = False


## 4. Optional: scan Drive for model/checkpoint files

Run this cell to see likely checkpoint or model files in your Drive.

In [5]:
def scan_drive_for_models(root="/content/drive/MyDrive", max_results=100):
    patterns = [
        "**/*.keras",
        "**/*.h5",
        "**/*.weights.h5",
        "**/*checkpoint*",
        "**/*weights*",
    ]
    hits = []
    for pattern in patterns:
        hits.extend(glob.glob(os.path.join(root, pattern), recursive=True))
    # De-duplicate and filter out very small/non-files
    unique = []
    seen = set()
    for p in hits:
        if p not in seen and os.path.isfile(p):
            seen.add(p)
            unique.append(p)
    rows = []
    for p in unique[:max_results]:
        rows.append({
            "path": p,
            "size_mb": os.path.getsize(p) / (1024**2),
            "modified": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(os.path.getmtime(p)))
        })
    return pd.DataFrame(rows).sort_values("modified", ascending=False) if rows else pd.DataFrame()

model_files_df = scan_drive_for_models()
display(model_files_df)

""


## 5. Model definitions

These definitions match the model descriptions in your report:

- CNN v3 / LWCNN: ShuffleNet-style lightweight CNN with depthwise separable blocks.
- CNN-Transformer: CNN frontend followed by Transformer encoder blocks.

If your saved checkpoint was trained with slightly different hyperparameters, loading weights may fail.  
That is okay for speed benchmarking: leave weights unloaded and use the architecture benchmark, or edit the config values below.

In [6]:
from tensorflow.keras import layers, models

# ---------------------------------------------------------------------
# CNN v3 / LWCNN
# ---------------------------------------------------------------------

CNN_V3_CONFIG = {
    "block_filters": 96,
    "final_filters": 512,
    "head_units": 128,
    "dropout_rate": 0.20,
}

# If your final CNN v3 was the 64-filter / 0.30-dropout version,
# uncomment this:
# CNN_V3_CONFIG = {
#     "block_filters": 64,
#     "final_filters": 512,
#     "head_units": 128,
#     "dropout_rate": 0.30,
# }

def dual_branch_block(x, filters, name="dual_branch"):
    # Left branch: depthwise stride-2 + pointwise projection
    left = layers.DepthwiseConv2D(
        kernel_size=3, strides=2, padding="same", use_bias=False,
        name=f"{name}_left_dw"
    )(x)
    left = layers.BatchNormalization(name=f"{name}_left_dw_bn")(left)
    left = layers.Conv2D(
        filters, kernel_size=1, padding="same", use_bias=False,
        name=f"{name}_left_pw"
    )(left)
    left = layers.BatchNormalization(name=f"{name}_left_pw_bn")(left)
    left = layers.ReLU(name=f"{name}_left_relu")(left)

    # Right branch: pointwise + depthwise stride-2 + pointwise
    right = layers.Conv2D(
        filters, kernel_size=1, padding="same", use_bias=False,
        name=f"{name}_right_pw1"
    )(x)
    right = layers.BatchNormalization(name=f"{name}_right_pw1_bn")(right)
    right = layers.ReLU(name=f"{name}_right_relu1")(right)
    right = layers.DepthwiseConv2D(
        kernel_size=3, strides=2, padding="same", use_bias=False,
        name=f"{name}_right_dw"
    )(right)
    right = layers.BatchNormalization(name=f"{name}_right_dw_bn")(right)
    right = layers.Conv2D(
        filters, kernel_size=1, padding="same", use_bias=False,
        name=f"{name}_right_pw2"
    )(right)
    right = layers.BatchNormalization(name=f"{name}_right_pw2_bn")(right)
    right = layers.ReLU(name=f"{name}_right_relu2")(right)

    return layers.Concatenate(name=f"{name}_concat")([left, right])

def channel_split_block(x, filters, name="channel_split"):
    # Split channels into identity and transformed halves
    c = x.shape[-1]
    if c is None:
        raise ValueError("Channel dimension must be known.")
    half = int(c) // 2

    left = layers.Lambda(lambda z: z[..., :half], name=f"{name}_identity_slice")(x)
    right = layers.Lambda(lambda z: z[..., half:], name=f"{name}_transform_slice")(x)

    right = layers.Conv2D(
        filters, kernel_size=1, padding="same", use_bias=False,
        name=f"{name}_pw1"
    )(right)
    right = layers.BatchNormalization(name=f"{name}_pw1_bn")(right)
    right = layers.ReLU(name=f"{name}_relu1")(right)

    right = layers.DepthwiseConv2D(
        kernel_size=3, padding="same", use_bias=False,
        name=f"{name}_dw"
    )(right)
    right = layers.BatchNormalization(name=f"{name}_dw_bn")(right)

    right = layers.Conv2D(
        filters, kernel_size=1, padding="same", use_bias=False,
        name=f"{name}_pw2"
    )(right)
    right = layers.BatchNormalization(name=f"{name}_pw2_bn")(right)
    right = layers.ReLU(name=f"{name}_relu2")(right)

    return layers.Concatenate(name=f"{name}_concat")([left, right])

def build_cnn_v3_lwcnn(input_shape=INPUT_SHAPE, config=CNN_V3_CONFIG):
    block_filters = config["block_filters"]
    final_filters = config["final_filters"]
    head_units = config["head_units"]
    dropout_rate = config["dropout_rate"]

    inputs = layers.Input(shape=input_shape, name="input_logmel")

    x = layers.Conv2D(
        block_filters, kernel_size=3, padding="same", use_bias=False,
        name="stem_conv"
    )(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.ReLU(name="stem_relu")(x)
    x = layers.MaxPooling2D(pool_size=2, strides=2, name="stem_pool")(x)

    x = dual_branch_block(x, block_filters, name="dual_branch_1")
    x = channel_split_block(x, block_filters, name="channel_split_1")

    x = layers.Conv2D(
        final_filters, kernel_size=1, padding="same", use_bias=False,
        name="final_projection"
    )(x)
    x = layers.BatchNormalization(name="final_projection_bn")(x)
    x = layers.ReLU(name="final_projection_relu")(x)

    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dropout(dropout_rate, name="dropout_1")(x)
    x = layers.Dense(head_units, activation="relu", name="dense_head")(x)
    x = layers.Dropout(dropout_rate, name="dropout_2")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="drone_probability")(x)

    return models.Model(inputs, outputs, name="CNN_v3_LWCNN")


# ---------------------------------------------------------------------
# CNN-Transformer
# ---------------------------------------------------------------------

CNN_TRANSFORMER_CONFIG = {
    "d_model": 96,
    "num_heads": 4,
    "num_layers": 2,
    "ffn_dim": 512,
    "dropout_rate": 0.40,
    "head_units": 64,
}

def transformer_encoder_block(x, d_model, num_heads, ffn_dim, dropout_rate, name="transformer"):
    attn_input = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_attn_ln")(x)
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout_rate,
        name=f"{name}_mha"
    )(attn_input, attn_input)
    attn_output = layers.Dropout(dropout_rate, name=f"{name}_attn_dropout")(attn_output)
    x = layers.Add(name=f"{name}_attn_residual")([x, attn_output])

    ffn_input = layers.LayerNormalization(epsilon=1e-6, name=f"{name}_ffn_ln")(x)
    ffn = layers.Dense(ffn_dim, activation="relu", name=f"{name}_ffn_dense1")(ffn_input)
    ffn = layers.Dropout(dropout_rate, name=f"{name}_ffn_dropout1")(ffn)
    ffn = layers.Dense(d_model, name=f"{name}_ffn_dense2")(ffn)
    ffn = layers.Dropout(dropout_rate, name=f"{name}_ffn_dropout2")(ffn)
    x = layers.Add(name=f"{name}_ffn_residual")([x, ffn])
    return x

def build_cnn_transformer(input_shape=INPUT_SHAPE, config=CNN_TRANSFORMER_CONFIG):
    d_model = config["d_model"]
    num_heads = config["num_heads"]
    num_layers = config["num_layers"]
    ffn_dim = config["ffn_dim"]
    dropout_rate = config["dropout_rate"]
    head_units = config["head_units"]

    inputs = layers.Input(shape=input_shape, name="input_logmel")

    x = layers.Conv2D(64, kernel_size=3, padding="same", use_bias=False, name="cnn_conv1")(inputs)
    x = layers.BatchNormalization(name="cnn_conv1_bn")(x)
    x = layers.ReLU(name="cnn_conv1_relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 2), name="cnn_pool1")(x)

    x = layers.SeparableConv2D(128, kernel_size=3, padding="same", use_bias=False, name="cnn_sepconv2")(x)
    x = layers.BatchNormalization(name="cnn_sepconv2_bn")(x)
    x = layers.ReLU(name="cnn_sepconv2_relu")(x)
    x = layers.MaxPooling2D(pool_size=(2, 1), name="cnn_pool2")(x)

    x = layers.SeparableConv2D(d_model, kernel_size=3, padding="same", use_bias=False, name="cnn_sepconv3")(x)
    x = layers.BatchNormalization(name="cnn_sepconv3_bn")(x)
    x = layers.ReLU(name="cnn_sepconv3_relu")(x)

    # Collapse frequency axis: (batch, freq, time, channels) -> (batch, time, channels)
    x = layers.Lambda(lambda z: tf.reduce_mean(z, axis=1), name="frequency_average_pooling")(x)

    # Convolutional positional encoding
    pos = layers.Conv1D(d_model, kernel_size=3, padding="same", name="conv_positional_encoding")(x)
    x = layers.Add(name="add_positional_encoding")([x, pos])

    for i in range(num_layers):
        x = transformer_encoder_block(
            x,
            d_model=d_model,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            dropout_rate=dropout_rate,
            name=f"transformer_block_{i+1}"
        )

    x = layers.LayerNormalization(epsilon=1e-6, name="final_layer_norm")(x)
    x = layers.GlobalAveragePooling1D(name="time_average_pooling")(x)
    x = layers.Dropout(dropout_rate, name="classifier_dropout_1")(x)
    x = layers.Dense(head_units, activation="relu", name="classifier_dense")(x)
    x = layers.Dropout(dropout_rate, name="classifier_dropout_2")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="drone_probability")(x)

    return models.Model(inputs, outputs, name="CNN_Transformer")

## 6. Benchmark helpers

In [7]:
def file_size_mb(path):
    if path is None or not os.path.exists(path):
        return np.nan
    return os.path.getsize(path) / (1024 ** 2)

def count_params(model):
    trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
    non_trainable = int(np.sum([np.prod(v.shape) for v in model.non_trainable_weights]))
    return trainable, non_trainable, trainable + non_trainable

def save_and_measure_model_size(model, name):
    out_dir = Path("/content/benchmark_saved_models")
    out_dir.mkdir(exist_ok=True)
    path = out_dir / f"{name}.keras"
    try:
        model.save(path)
        return str(path), file_size_mb(str(path))
    except Exception as e:
        print(f"Could not save {name}: {e}")
        return None, np.nan

def load_optional_weights(model, weights_path):
    if weights_path is None:
        return "not_requested"
    if not os.path.exists(weights_path):
        return f"not_found: {weights_path}"
    try:
        model.load_weights(weights_path)
        return f"loaded: {weights_path}"
    except Exception as e:
        return f"failed_to_load: {type(e).__name__}: {e}"

def load_full_model_or_build(name, full_model_path, build_fn):
    if full_model_path is not None and os.path.exists(full_model_path):
        model = tf.keras.models.load_model(full_model_path, compile=False)
        source = f"full_model_loaded: {full_model_path}"
    else:
        model = build_fn()
        source = "architecture_built_in_notebook"
    return model, source

def benchmark_model(model, input_shape, batch_size=1, warmup_runs=30, measure_runs=300, window_duration_seconds=0.5):
    x = tf.random.normal((batch_size, *input_shape), dtype=tf.float32)

    # Build/trace once
    _ = model(x, training=False)

    for _ in range(warmup_runs):
        _ = model(x, training=False).numpy()

    latencies = []
    for _ in range(measure_runs):
        start = time.perf_counter()
        y = model(x, training=False)
        _ = y.numpy()  # forces GPU synchronisation
        end = time.perf_counter()
        latencies.append((end - start) / batch_size)

    latencies = np.asarray(latencies)
    mean_s = float(np.mean(latencies))
    median_s = float(np.median(latencies))
    p95_s = float(np.percentile(latencies, 95))
    p99_s = float(np.percentile(latencies, 99))

    return {
        "batch_size": batch_size,
        "mean_latency_ms_per_window": mean_s * 1000,
        "median_latency_ms_per_window": median_s * 1000,
        "p95_latency_ms_per_window": p95_s * 1000,
        "p99_latency_ms_per_window": p99_s * 1000,
        "windows_per_second_mean": 1.0 / mean_s if mean_s > 0 else np.nan,
        "real_time_factor_mean": mean_s / window_duration_seconds if window_duration_seconds > 0 else np.nan,
        "times_faster_than_real_time": window_duration_seconds / mean_s if mean_s > 0 else np.nan,
    }

def benchmark_entry(model_name, build_fn, full_model_path=None, weights_path=None):
    print(f"\n=== {model_name} ===")
    model, model_source = load_full_model_or_build(model_name, full_model_path, build_fn)
    weight_status = "not_applicable_full_model"
    if model_source == "architecture_built_in_notebook":
        weight_status = load_optional_weights(model, weights_path)

    model.summary()

    trainable, non_trainable, total = count_params(model)
    saved_path, saved_size_mb = save_and_measure_model_size(
        model,
        model_name.lower().replace(" ", "_").replace("/", "_")
    )

    bench = benchmark_model(
        model=model,
        input_shape=INPUT_SHAPE,
        batch_size=BATCH_SIZE,
        warmup_runs=WARMUP_RUNS,
        measure_runs=MEASURE_RUNS,
        window_duration_seconds=WINDOW_DURATION_SECONDS,
    )

    return {
        "model": model_name,
        "model_source": model_source,
        "weights_status": weight_status,
        "input_shape": str(INPUT_SHAPE),
        "trainable_parameters": trainable,
        "non_trainable_parameters": non_trainable,
        "total_parameters": total,
        "checkpoint_or_model_path": full_model_path if full_model_path else weights_path,
        "checkpoint_or_model_size_mb": file_size_mb(full_model_path if full_model_path else weights_path),
        "saved_keras_path": saved_path,
        "saved_keras_size_mb": saved_size_mb,
        **bench,
    }

## 7. Run the benchmark

This is the main cell. It benchmarks both models and saves the results.

In [8]:
results = []

results.append(
    benchmark_entry(
        model_name="CNN v3 / LWCNN",
        build_fn=lambda: build_cnn_v3_lwcnn(INPUT_SHAPE, CNN_V3_CONFIG),
        full_model_path=CNN_V3_FULL_MODEL_PATH,
        weights_path=CNN_V3_WEIGHTS_PATH,
    )
)

results.append(
    benchmark_entry(
        model_name="CNN-Transformer",
        build_fn=lambda: build_cnn_transformer(INPUT_SHAPE, CNN_TRANSFORMER_CONFIG),
        full_model_path=CNN_TRANSFORMER_FULL_MODEL_PATH,
        weights_path=CNN_TRANSFORMER_WEIGHTS_PATH,
    )
)

df = pd.DataFrame(results)
display(df)


=== CNN v3 / LWCNN ===


Model: "CNN_v3_LWCNN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_logmel        │ (None, 64, 63, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 64, 63,    │        864 │ input_logmel[0][… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 64, 63,    │        384 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu (ReLU)    │ (None, 64, 63,    │          0 │ stem_bn[0][0]     │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_pool           │ (None, 32, 31,    │          0 │ stem_relu[0][0]   │
│ (MaxPooling2D)      │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_righ… │ (None, 32, 31,    │      9,216 │ stem_pool[0][0]   │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_righ… │ (None, 32, 31,    │        384 │ dual_branch_1_ri… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_righ… │ (None, 32, 31,    │          0 │ dual_branch_1_ri… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_left… │ (None, 16, 16,    │        864 │ stem_pool[0][0]   │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_righ… │ (None, 16, 16,    │        864 │ dual_branch_1_ri… │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_left… │ (None, 16, 16,    │        384 │ dual_branch_1_le… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_righ… │ (None, 16, 16,    │        384 │ dual_branch_1_ri… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_left… │ (None, 16, 16,    │      9,216 │ dual_branch_1_le… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_righ… │ (None, 16, 16,    │      9,216 │ dual_branch_1_ri… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_left… │ (None, 16, 16,    │        384 │ dual_branch_1_le… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_righ… │ (None, 16, 16,    │        384 │ dual_branch_1_ri… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_branch_1_left… │ (None, 16, 16,    │          0 │ dual_branch_1_le

 Total params: 219,137 (856.00 KB)

 Trainable params: 216,385 (845.25 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_logmel        │ (None, 64, 63, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_conv1 (Conv2D)  │ (None, 64, 63,    │        576 │ input_logmel[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_conv1_bn        │ (None, 64, 63,    │        256 │ cnn_conv1[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_conv1_relu      │ (None, 64, 63,    │          0 │ cnn_conv1_bn[0][… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_pool1           │ (None, 32, 31,    │          0 │ cnn_conv1_relu[0… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_sepconv2        │ (None, 32, 31,    │      8,768 │ cnn_pool1[0][0]   │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_sepconv2_bn     │ (None, 32, 31,    │        512 │ cnn_sepconv2[0][… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_sepconv2_relu   │ (None, 32, 31,    │          0 │ cnn_sepconv2_bn[… │
│ (ReLU)              │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_pool2           │ (None, 16, 31,    │          0 │ cnn_sepconv2_rel… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_sepconv3        │ (None, 16, 31,    │     13,440 │ cnn_pool2[0][0]   │
│ (SeparableConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_sepconv3_bn     │ (None, 16, 31,    │        384 │ cnn_sepconv3[0][… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_sepconv3_relu   │ (None, 16, 31,    │          0 │ cnn_sepconv3_bn[… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ frequency_average_… │ (None, 31, 96)    │          0 │ cnn_sepconv3_rel… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_positional_en… │ (None, 31, 96)    │     27,744 │ frequency_averag… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_positional_enc… │ (None, 31, 96)    │          0 │ frequency_averag… │
│ (Add)               │                   │            │ conv_positional_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_… │ (None, 31, 96)    │        192 │ add_positional_e… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_… │ (None, 31, 96)    │     37,248 │ transformer_bloc

 Total params: 331,233 (1.26 MB)

 Trainable params: 330,657 (1.26 MB)

 Non-trainable params: 576 (2.25 KB)

,model,model_source,weights_status,input_shape,trainable_parameters,non_trainable_parameters,total_parameters,checkpoint_or_model_path,checkpoint_or_model_size_mb,saved_keras_path,saved_keras_size_mb,batch_size,mean_latency_ms_per_window,median_latency_ms_per_window,p95_latency_ms_per_window,p99_latency_ms_per_window,windows_per_second_mean,real_time_factor_mean,times_faster_than_real_time
0,CNN v3 / LWCNN,architecture_built_in_notebook,not_found: /content/drive/MyDrive/lwcnn_ext_se...,"(64, 63, 1)",216385,2752,219137,/content/drive/MyDrive/lwcnn_ext_search/trial_...,NaN,/content/benchmark_saved_models/cnn_v3___lwcnn...,0.968029,1,45.485557,44.680217,52.059573,55.250248,21.985001,0.090971,10.992500
1,CNN-Transformer,architecture_built_in_notebook,not_found: /content/drive/MyDrive/cnntransform...,"(64, 63, 1)",330657,576,331233,/content/drive/MyDrive/cnntransformer_ext_sear...,NaN,/content/benchmark_saved_models/cnn-transforme...,1.423329,1,65.656942,64.703553,73.274176,79.603606,15.230682,0.131314,7.615341


## 7A. CNN-Transformer internal/external F1 evaluation

This optional section evaluates the trained CNN-Transformer checkpoint on internal and external held-out sets.

Set the paths in the configuration cell above, then run this section after the benchmark cell. The code assumes binary labels where `1 = Drone` and `0 = No-drone`. If your saved labels are reversed, set the relevant `CNN_TRANSFORMER_FLIP_*_LABELS` flag to `True`.


In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


def _first_existing_key(npz_obj, candidates):
    for key in candidates:
        if key in npz_obj.files:
            return key
    return None


def _load_npz_xy(npz_path):
    data = np.load(npz_path, allow_pickle=False)

    x_candidates = [
        "X", "x", "X_test", "x_test", "features", "spectrograms",
        "logmel", "log_mel", "mel", "inputs", "data", "arr_0"
    ]
    y_candidates = [
        "y", "Y", "y_test", "labels", "targets", "target", "classes", "arr_1"
    ]

    x_key = _first_existing_key(data, x_candidates)
    y_key = _first_existing_key(data, y_candidates)

    # If keys are not standard, infer from array shapes.
    if x_key is None or y_key is None:
        arrays = {key: data[key] for key in data.files}
        feature_like = [key for key, arr in arrays.items() if np.asarray(arr).ndim >= 2]
        label_like = [key for key, arr in arrays.items() if np.asarray(arr).ndim <= 2]
        if x_key is None and feature_like:
            x_key = feature_like[0]
        if y_key is None:
            for key in label_like:
                if key != x_key:
                    y_key = key
                    break

    if x_key is None or y_key is None:
        raise ValueError(
            f"Could not identify feature/label arrays in {npz_path}. "
            f"Available keys: {data.files}"
        )

    return np.asarray(data[x_key]), np.asarray(data[y_key]), f"npz:{npz_path} (X={x_key}, y={y_key})"


def _prepare_eval_arrays(X, y, input_shape=INPUT_SHAPE, flip_labels=False):
    X = np.asarray(X)
    y = np.asarray(y)
    if y.ndim == 2 and y.shape[1] > 1:
        y = np.argmax(y, axis=1)
    y = y.reshape(-1)

    # Accept features saved as (N, H, W) by adding the channel dimension.
    if X.ndim == 3 and len(input_shape) == 3 and input_shape[-1] == 1:
        X = X[..., np.newaxis]

    if X.shape[1:] != tuple(input_shape):
        raise ValueError(
            f"Evaluation feature shape {X.shape[1:]} does not match INPUT_SHAPE {input_shape}. "
            "Update INPUT_SHAPE or load the matching preprocessed test set."
        )

    if len(X) != len(y):
        raise ValueError(f"Number of samples and labels differ: X={len(X)}, y={len(y)}")

    y = (y.astype(float) >= 0.5).astype(int)
    if flip_labels:
        y = 1 - y

    return X.astype(np.float32), y.astype(int)


def load_eval_split(split_name, npz_path=None, x_path=None, y_path=None, data_dir=None, flip_labels=False):
    if npz_path:
        X, y, status = _load_npz_xy(npz_path)
        X, y = _prepare_eval_arrays(X, y, INPUT_SHAPE, flip_labels=flip_labels)
        return {"kind": "arrays", "X": X, "y": y, "status": status}

    if x_path and y_path:
        X = np.load(x_path, allow_pickle=False)
        y = np.load(y_path, allow_pickle=False)
        X, y = _prepare_eval_arrays(X, y, INPUT_SHAPE, flip_labels=flip_labels)
        return {"kind": "arrays", "X": X, "y": y, "status": f"npy:X={x_path}, y={y_path}"}

    if data_dir:
        ds = tf.keras.utils.image_dataset_from_directory(
            data_dir,
            labels="inferred",
            label_mode="int",
            color_mode="grayscale",
            batch_size=CNN_TRANSFORMER_EVAL_BATCH_SIZE,
            image_size=INPUT_SHAPE[:2],
            shuffle=False,
        )
        class_names = list(ds.class_names)
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
        if flip_labels:
            ds = ds.map(lambda x, y: (x, 1 - tf.cast(tf.reshape(y, [-1]), tf.int32)))
        return {"kind": "dataset", "dataset": ds, "class_names": class_names, "status": f"image_directory:{data_dir}"}

    return {"kind": "missing", "status": f"skipped_no_{split_name}_data_path_configured"}


def evaluate_binary_model_on_split(model, split, split_name, threshold=CNN_TRANSFORMER_DECISION_THRESHOLD):
    if split["kind"] == "missing":
        return {
            "split": split_name,
            "status": split["status"],
            "n_samples": 0,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1_binary": np.nan,
            "f1_macro": np.nan,
            "threshold": threshold,
            "confusion_matrix": None,
        }

    if split["kind"] == "arrays":
        X = split["X"]
        y_true = split["y"]
        y_prob = model.predict(X, batch_size=CNN_TRANSFORMER_EVAL_BATCH_SIZE, verbose=0).reshape(-1)
    elif split["kind"] == "dataset":
        y_true_batches = []
        for _, y_batch in split["dataset"]:
            y_true_batches.append(np.asarray(y_batch).reshape(-1))
        y_true = np.concatenate(y_true_batches).astype(int)
        y_prob = model.predict(split["dataset"], verbose=0).reshape(-1)
    else:
        raise ValueError(f"Unknown split kind: {split['kind']}")

    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print("\n" + "="*80)
    print(f"CNN-Transformer {split_name} evaluation")
    print("="*80)
    print("Data source:", split["status"])
    if split.get("class_names"):
        print("Class names from directory:", split["class_names"])
        print("Note: binary F1 assumes model output probability corresponds to label 1.")
    print(f"Threshold: {threshold:.2f}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["No-drone", "Drone"],
        zero_division=0,
    ))

    return {
        "split": split_name,
        "status": split["status"],
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1_binary": float(f1_score(y_true, y_pred, zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "threshold": float(threshold),
        "confusion_matrix": cm.tolist(),
    }


# Build/load the CNN-Transformer exactly as in the benchmark, then evaluate it.
cnn_transformer_model, cnn_transformer_source = load_full_model_or_build(
    "CNN-Transformer",
    CNN_TRANSFORMER_FULL_MODEL_PATH,
    lambda: build_cnn_transformer(INPUT_SHAPE, CNN_TRANSFORMER_CONFIG),
)

cnn_transformer_weight_status = "not_applicable_full_model"
if cnn_transformer_source == "architecture_built_in_notebook":
    cnn_transformer_weight_status = load_optional_weights(cnn_transformer_model, CNN_TRANSFORMER_WEIGHTS_PATH)

print("CNN-Transformer source:", cnn_transformer_source)
print("CNN-Transformer weights:", cnn_transformer_weight_status)

internal_split = load_eval_split(
    "internal",
    npz_path=CNN_TRANSFORMER_INTERNAL_TEST_NPZ,
    x_path=CNN_TRANSFORMER_INTERNAL_X_PATH,
    y_path=CNN_TRANSFORMER_INTERNAL_Y_PATH,
    data_dir=CNN_TRANSFORMER_INTERNAL_DATA_DIR,
    flip_labels=CNN_TRANSFORMER_FLIP_INTERNAL_LABELS,
)

external_split = load_eval_split(
    "external",
    npz_path=CNN_TRANSFORMER_EXTERNAL_TEST_NPZ,
    x_path=CNN_TRANSFORMER_EXTERNAL_X_PATH,
    y_path=CNN_TRANSFORMER_EXTERNAL_Y_PATH,
    data_dir=CNN_TRANSFORMER_EXTERNAL_DATA_DIR,
    flip_labels=CNN_TRANSFORMER_FLIP_EXTERNAL_LABELS,
)

cnn_transformer_f1_results = [
    evaluate_binary_model_on_split(cnn_transformer_model, internal_split, "internal"),
    evaluate_binary_model_on_split(cnn_transformer_model, external_split, "external"),
]

cnn_transformer_f1_df = pd.DataFrame(cnn_transformer_f1_results)
display(cnn_transformer_f1_df)

# Add the F1 scores to the main benchmark dataframe so they are saved with the rest of the results.
for col in [
    "internal_f1_score", "external_f1_score",
    "internal_f1_macro", "external_f1_macro",
    "internal_accuracy", "external_accuracy",
    "internal_precision", "external_precision",
    "internal_recall", "external_recall",
    "internal_n_samples", "external_n_samples",
    "internal_eval_status", "external_eval_status",
]:
    if col not in df.columns:
        df[col] = np.nan if not col.endswith("status") else ""

for result in cnn_transformer_f1_results:
    prefix = result["split"]
    row_mask = df["model"].eq("CNN-Transformer")
    df.loc[row_mask, f"{prefix}_f1_score"] = result["f1_binary"]
    df.loc[row_mask, f"{prefix}_f1_macro"] = result["f1_macro"]
    df.loc[row_mask, f"{prefix}_accuracy"] = result["accuracy"]
    df.loc[row_mask, f"{prefix}_precision"] = result["precision"]
    df.loc[row_mask, f"{prefix}_recall"] = result["recall"]
    df.loc[row_mask, f"{prefix}_n_samples"] = result["n_samples"]
    df.loc[row_mask, f"{prefix}_eval_status"] = result["status"]

print("Updated benchmark table with CNN-Transformer internal/external F1 columns:")
display(df)

# Keep the JSON-export list synchronised with the updated dataframe.
results = df.to_dict(orient="records")


CNN-Transformer source: architecture_built_in_notebook
CNN-Transformer weights: not_found: /content/drive/MyDrive/cnntransformer_ext_search/trial_4/checkpoint.weights.h5


,split,status,n_samples,accuracy,precision,recall,f1_binary,f1_macro,threshold,confusion_matrix
0,internal,skipped_no_internal_data_path_configured,0,NaN,NaN,NaN,NaN,NaN,0.5,None
1,external,skipped_no_external_data_path_configured,0,NaN,NaN,NaN,NaN,NaN,0.5,None


Updated benchmark table with CNN-Transformer internal/external F1 columns:


,model,model_source,weights_status,input_shape,trainable_parameters,non_trainable_parameters,total_parameters,checkpoint_or_model_path,checkpoint_or_model_size_mb,saved_keras_path,...,internal_accuracy,external_accuracy,internal_precision,external_precision,internal_recall,external_recall,internal_n_samples,external_n_samples,internal_eval_status,external_eval_status
0,CNN v3 / LWCNN,architecture_built_in_notebook,not_found: /content/drive/MyDrive/lwcnn_ext_se...,"(64, 63, 1)",216385,2752,219137,/content/drive/MyDrive/lwcnn_ext_search/trial_...,NaN,/content/benchmark_saved_models/cnn_v3___lwcnn...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,,
1,CNN-Transformer,architecture_built_in_notebook,not_found: /content/drive/MyDrive/cnntransform...,"(64, 63, 1)",330657,576,331233,/content/drive/MyDrive/cnntransformer_ext_sear...,NaN,/content/benchmark_saved_models/cnn-transforme...,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,skipped_no_internal_data_path_configured,skipped_no_external_data_path_configured


## 8. Save benchmark results

In [10]:
csv_path = "/content/cnn_v3_cnn_transformer_benchmark_results.csv"
json_path = "/content/cnn_v3_cnn_transformer_benchmark_results.json"

df.to_csv(csv_path, index=False)
with open(json_path, "w") as f:
    json.dump(df.to_dict(orient="records"), f, indent=2)

print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)

# Save the optional CNN-Transformer F1 results separately as well.
if "cnn_transformer_f1_df" in globals():
    f1_csv_path = "/content/cnn_transformer_internal_external_f1_results.csv"
    f1_json_path = "/content/cnn_transformer_internal_external_f1_results.json"
    cnn_transformer_f1_df.to_csv(f1_csv_path, index=False)
    with open(f1_json_path, "w") as f:
        json.dump(cnn_transformer_f1_results, f, indent=2)
    print("Saved F1 CSV:", f1_csv_path)
    print("Saved F1 JSON:", f1_json_path)


Saved CSV: /content/cnn_v3_cnn_transformer_benchmark_results.csv
Saved JSON: /content/cnn_v3_cnn_transformer_benchmark_results.json
Saved F1 CSV: /content/cnn_transformer_internal_external_f1_results.csv
Saved F1 JSON: /content/cnn_transformer_internal_external_f1_results.json


## 9. Report-ready paragraph generator

This produces wording you can paste into the report.

In [11]:
def fmt_int(x):
    return f"{int(x):,}"

def fmt(x, nd=2):
    if pd.isna(x):
        return "N/A"
    return f"{float(x):.{nd}f}"

for _, r in df.iterrows():
    print("\n" + "="*80)
    print(r["model"])
    print("="*80)

    paragraph = (
        f"The {r['model']} contained {fmt_int(r['trainable_parameters'])} trainable parameters "
        f"({fmt_int(r['total_parameters'])} total parameters). When saved as a Keras model, "
        f"it had a disk footprint of {fmt(r['saved_keras_size_mb'])} MB. "
        f"Using a batch size of {int(r['batch_size'])} on the selected Colab runtime, "
        f"the model achieved a mean inference latency of {fmt(r['mean_latency_ms_per_window'])} ms "
        f"per 0.5-second audio window, corresponding to a real-time factor of approximately "
        f"{fmt(r['real_time_factor_mean'], 3)}×. This means the model processed audio "
        f"approximately {fmt(r['times_faster_than_real_time'])} times faster than real time."
    )

    if r["model"] == "CNN-Transformer" and "internal_f1_score" in df.columns and "external_f1_score" in df.columns:
        if not pd.isna(r.get("internal_f1_score", np.nan)) or not pd.isna(r.get("external_f1_score", np.nan)):
            paragraph += (
                f" The CNN-Transformer achieved an internal F1 score of "
                f"{fmt(r.get('internal_f1_score', np.nan), 3)} and an external F1 score of "
                f"{fmt(r.get('external_f1_score', np.nan), 3)} at a decision threshold of "
                f"{fmt(CNN_TRANSFORMER_DECISION_THRESHOLD, 2)}."
            )

    print(paragraph)



CNN v3 / LWCNN
The CNN v3 / LWCNN contained 216,385 trainable parameters (219,137 total parameters). When saved as a Keras model, it had a disk footprint of 0.97 MB. Using a batch size of 1 on the selected Colab runtime, the model achieved a mean inference latency of 45.49 ms per 0.5-second audio window, corresponding to a real-time factor of approximately 0.091×. This means the model processed audio approximately 10.99 times faster than real time.

CNN-Transformer
The CNN-Transformer contained 330,657 trainable parameters (331,233 total parameters). When saved as a Keras model, it had a disk footprint of 1.42 MB. Using a batch size of 1 on the selected Colab runtime, the model achieved a mean inference latency of 65.66 ms per 0.5-second audio window, corresponding to a real-time factor of approximately 0.131×. This means the model processed audio approximately 7.62 times faster than real time.


## 10. Optional: batch-size comparison

For the report, use **batch size 1** because real-time streaming usually processes one window at a time.  
This optional section shows how throughput changes when processing multiple windows at once.

In [ ]:
BATCH_SIZES_TO_TEST = [1, 8, 16, 32]

batch_results = []
for model_name, build_fn, full_model_path, weights_path in [
    ("CNN v3 / LWCNN", lambda: build_cnn_v3_lwcnn(INPUT_SHAPE, CNN_V3_CONFIG), CNN_V3_FULL_MODEL_PATH, CNN_V3_WEIGHTS_PATH),
    ("CNN-Transformer", lambda: build_cnn_transformer(INPUT_SHAPE, CNN_TRANSFORMER_CONFIG), CNN_TRANSFORMER_FULL_MODEL_PATH, CNN_TRANSFORMER_WEIGHTS_PATH),
]:
    model, model_source = load_full_model_or_build(model_name, full_model_path, build_fn)
    if model_source == "architecture_built_in_notebook":
        _ = load_optional_weights(model, weights_path)

    trainable, non_trainable, total = count_params(model)
    for bs in BATCH_SIZES_TO_TEST:
        bench = benchmark_model(
            model=model,
            input_shape=INPUT_SHAPE,
            batch_size=bs,
            warmup_runs=20,
            measure_runs=100,
            window_duration_seconds=WINDOW_DURATION_SECONDS,
        )
        batch_results.append({
            "model": model_name,
            "batch_size": bs,
            "trainable_parameters": trainable,
            "total_parameters": total,
            **bench
        })

batch_df = pd.DataFrame(batch_results)
display(batch_df)

batch_csv_path = "/content/cnn_v3_cnn_transformer_batchsize_benchmark_results.csv"
batch_df.to_csv(batch_csv_path, index=False)
print("Saved:", batch_csv_path)

## Reporting note

In the report, describe the benchmark setup clearly. For example:

> Inference benchmarking was conducted in Google Colab on the available GPU runtime using TensorFlow, with batch size 1. Each model was evaluated on synthetic log-mel inputs matching the model input shape, with 30 warm-up passes followed by 300 measured forward passes. Latency therefore measures model inference only, excluding audio loading and log-mel feature extraction.

If you want full pipeline latency, add timing around audio loading, resampling, log-mel generation, and model inference together.